In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import os

input_dir = "/kaggle/input/"
master_file = None
train_file = None

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if "master_dataset_ready" in file:
            master_file = os.path.join(root, file)
        elif "500_Train_Tweet" in file:
            train_file = os.path.join(root, file)

def load_data(file_path):
    if file_path.endswith('.xlsx'):
        df = pd.read_excel(file_path, dtype=str)
    else:
        df = pd.read_csv(file_path, dtype=str, engine='python', on_bad_lines='skip', encoding_errors='replace')

    df.columns = df.columns.str.lower().str.strip()
    return df

master_df = load_data(master_file)
train_df = load_data(train_file)


master_df['id'] = master_df['id'].fillna('1234567890123456789')
train_df['id'] = train_df['id'].fillna('1234567890123456789')

master_df['text'] = master_df['text'].fillna('[NO TEXT]')
train_df['text'] = train_df['text'].fillna('[NO TEXT]')

train_df = train_df.dropna(subset=['label'])
master_df['id'] = master_df['id'].astype(str).str.replace('tweet-', '', regex=False).str.strip()
train_df['id'] = train_df['id'].astype(str).str.replace('tweet-', '', regex=False).str.strip()
unlabeled_df = master_df[~master_df['id'].isin(train_df['id'])].copy()
unlabeled_df = unlabeled_df[unlabeled_df['text'] != '[NO TEXT]']

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TweetDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = TweetDataset(train_df['text'].astype(str).tolist(), train_df['label'].astype(int).tolist())
unlabeled_dataset = TweetDataset(unlabeled_df['text'].astype(str).tolist())

print(f"Setting up {MODEL_NAME} for 5 emotions...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print(f"Training on your {len(train_df)} labeled tweets...")
trainer.train()

print("Predicting the remaining tweets...")
predictions = trainer.predict(unlabeled_dataset)

probs = F.softmax(torch.tensor(predictions.predictions), dim=-1)
max_probs, predicted_labels = torch.max(probs, dim=-1)

unlabeled_df['Label'] = predicted_labels.numpy()
unlabeled_df['confidence'] = max_probs.numpy()

final_pseudo_df = unlabeled_df.copy()


print("Formatting the final output...")
final_pseudo_df = final_pseudo_df.rename(columns={'id': 'ID', 'text': 'Text'})
final_pseudo_df = final_pseudo_df[['ID', 'Text', 'Label']]

train_df_formatted = train_df.rename(columns={'id': 'ID', 'text': 'Text', 'label': 'Label'})
train_df_formatted = train_df_formatted[['ID', 'Text', 'Label']]

final_submission_df = pd.concat([train_df_formatted, final_pseudo_df], ignore_index=True)

output_path = "/kaggle/working/Final_Pseudolabeled_Dataset.xlsx"
final_submission_df.to_excel(output_path, index=False)
print(f"Done! Your final dataset is saved at {output_path}")

Setting up xlm-roberta-base for 5 emotions...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training on your 499 labeled tweets...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
10,3.271515
20,3.190543
30,3.061895
40,2.755303
50,2.494164
60,2.116757


Predicting the remaining tweets...


Formatting the final output...
Done! Your final dataset is saved at /kaggle/working/Final_Pseudolabeled_Dataset.xlsx
